# 3 — Isolated seed-999 end-to-end smoke gate
Run one complete candidate matrix at the primary horizon: 18 physical episodes (three shared ID tasks and six OOD candidates, each at Native/+200). Outputs stay under `~/stage3_new_smoke`.


In [ ]:
import csv, os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"
OUT=Path.home()/"stage3_new_smoke"; OUT.mkdir(exist_ok=True); MAN=OUT/"stage3_new_smoke_manifest.csv"; AUD=OUT/"stage3_new_smoke_pairing.csv"; GPU=(Path.home()/"stage3_new_gpu.txt").read_text().strip()
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(); plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.make_stage3_new_manifest","--output",str(MAN),"--smoke-seed","999","--smoke-horizon","25","--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,check=True)
base=os.environ.copy(); base.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1"})
for scene,py in (("id",ID),("ood",OOD)):
    env=base.copy()
    if scene=="ood": env.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
    subprocess.run([str(py),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3_new.yaml"),"--manifest",str(MAN),"--scene",scene,"--expected-rows","18","--expected-cells-per-key","2","--audit-output",str(AUD)],cwd=R,env=env,check=True)
    log=OUT/f"stage3_new_smoke_{scene}.log"; cmd=[str(py),"-u","-m","async_vla_benchmark.scripts.run_stage3_new","--config",str(R/"async_vla_benchmark/configs/stage3_new.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",scene,"--resume","--verbose"]
    with open(log,"ab") as fh: subprocess.run(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,check=True)
rows=list(csv.DictReader(open(OUT/"stage3_new_episode_results.csv"))); assert len(rows)==18 and {int(r['seed']) for r in rows}=={999} and all(r['status'].startswith('ok') for r in rows)
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.validate_stage3_new","--manifest",str(MAN),"--output-dir",str(OUT),"--smoke"],cwd=R,check=True)
print("PASS: 18 seed-999 smoke episodes; frozen seeds 46..109 untouched")
print("STOP HERE and inspect both log tails before notebook 04.")
